In [1]:
import importlib
import sys
import pickle

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')

from model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

In [2]:
#load model
file_path_model = '../../../training_variational_dropout_v2/Sepsis/Sepsis_full_grad_norm_new_v2.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(file_path_model, dropout=0.1)

# Load the dataset
file_path_data_set = '../../../../../encoded_data_v2/Sepsis/sepsis_all_5_test.pkl'
sepsis_test_dataset = torch.load(file_path_data_set, weights_only=False)

Dynamic data set categories:  ([('concept:name', 18, {'Admission IC': 1, 'Admission NC': 2, 'CRP': 3, 'EOS': 4, 'ER Registration': 5, 'ER Sepsis Triage': 6, 'ER Triage': 7, 'IV Antibiotics': 8, 'IV Liquid': 9, 'LacticAcid': 10, 'Leucocytes': 11, 'Release A': 12, 'Release B': 13, 'Release C': 14, 'Release D': 15, 'Release E': 16, 'Return ER': 17}), ('org:group', 27, {'?': 1, 'A': 2, 'B': 3, 'C': 4, 'D': 5, 'E': 6, 'EOS': 7, 'F': 8, 'G': 9, 'H': 10, 'I': 11, 'J': 12, 'K': 13, 'L': 14, 'M': 15, 'N': 16, 'O': 17, 'P': 18, 'Q': 19, 'R': 20, 'S': 21, 'T': 22, 'U': 23, 'V': 24, 'W': 25, 'X': 26}), ('lifecycle:transition', 3, {'EOS': 1, 'complete': 2})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {}), ('Leucocytes', 1, {}), ('CRP', 1, {}), ('LacticAcid', 1, {})])
Data set static categories:  ([('Age', 16, {20.0: 1, 25.0: 2, 30.0: 3, 35.0: 4, 40.0: 5, 45.0: 6, 50.0: 7, 55.0: 8, 60.0: 9, 65.0: 10, 70.0: 11, 75.0: 12, 80.0: 13, 85.0

In [3]:
import evaluation_v2.probabilistic_evaluation
importlib.reload(evaluation_v2.probabilistic_evaluation)
from evaluation_v2.probabilistic_evaluation import ProbabilisticEvaluation

new_eval = ProbabilisticEvaluation(model=model, 
                                   dataset=sepsis_test_dataset,
                                   concept_name='concept:name',
                                   num_processes=16,
                                   #growing_num_values = [],
                                   growing_num_values = ['case_elapsed_time'],
                                   # number of samples
                                   samples_per_case = 100,
                                   sample_argmax = False,
                                   use_variance_cat = True,
                                   use_variance_num = True,
                                   decoder_cat=['concept:name'],
                                   decoder_num=['case_elapsed_time', 'event_elapsed_time']
                                   )

In [4]:
def save_chunk(results, i):
    chunk_number = (i + 1)
    filename = os.path.join(output_dir, f'results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} results to {filename}")

output_dir = '../../../../../../../data/Sepsis/v2/'

save_every = 50

results = {}
#for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate(random_order=True)):
for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate_multi_processing(random_order=True)):
    # print(case_name, prefix_len)
    assert((case_name, prefix_len) not in results)
    results[(case_name, prefix_len)] = (prefix, suffix, mean_prediction, predicted_suffixes)
    # print(prefix_len, len(suffix))
    if (i + 1) % save_every == 0:
        save_chunk(results, i)
        results = {}

if len(results):
    save_chunk(results, i)

  0%|          | 0/209 [00:00<?, ?it/s]

Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_050.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_100.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_150.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_200.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_250.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_300.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_350.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_400.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_450.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_500.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_550.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_600.pkl
Saved 50 results to ../../../../../../../data/Sepsis/v2/results_part_650.pkl